# 🎶 Music Generation with Transformers

In this notebook, we'll walk through the steps required to train your own Transformer model to generate music in the style of the Bach cello suites

In [ ]:
%load_ext autoreload
%autoreload 2

# TensorBoard imports TensorFlow whenever it is installed, and TensorFlow and PyTorch
# in one process can crash. This makes TensorBoard use its own TF-free version instead
import sys, types
sys.modules["tensorboard.compat.notf"] = types.ModuleType("tensorboard.compat.notf")

import os
import glob
import math
import numpy as np
import time
import matplotlib.pyplot as plt
from collections import Counter

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, ConcatDataset
from transformers import Trainer, TrainingArguments, TrainerCallback

import music21

from transformer_utils_pytorch import (
    parse_midi_files,
    load_parsed_files,
    get_midi_note,
    SinePositionEncoding,
)

# Check GPU - the Hugging Face Trainer picks the best available device
# automatically, so we only report what it will find
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")

## 0. Parameters <a name="parameters"></a>

In [ ]:
PARSE_MIDI_FILES = True
PARSED_DATA_PATH = "./parsed_data/"
DATASET_REPETITIONS = 1

SEQ_LEN = 50
EMBEDDING_DIM = 256
KEY_DIM = 256
N_HEADS = 5
DROPOUT_RATE = 0.3
FEED_FORWARD_DIM = 256
LOAD_MODEL = False

# optimization
EPOCHS = 5000
BATCH_SIZE = 256

GENERATE_LEN = 50

## 1. Prepare the Data

In [ ]:
# Load the data
file_list = glob.glob("../../../data/bach-cello/*.mid")
print(f"Found {len(file_list)} midi files")

In [ ]:
parser = music21.converter

In [ ]:
example_score = (
    music21.converter.parse(file_list[1]).splitAtQuarterLength(12)[0].chordify()
)

In [ ]:
example_score.show()

In [ ]:
example_score.show("text")

In [ ]:
if PARSE_MIDI_FILES:
    notes, durations = parse_midi_files(
        file_list, parser, SEQ_LEN + 1, PARSED_DATA_PATH
    )
else:
    notes, durations = load_parsed_files(PARSED_DATA_PATH)

In [ ]:
example_notes = notes[658]
example_durations = durations[658]
print("\nNotes string\n", example_notes, "...")
print("\nDuration string\n", example_durations, "...")

## 2. Tokenize the data <a name="tokenize"></a>

In [ ]:
def create_dataset(elements):
    # A simple replacement for Keras' TextVectorization layer: split on spaces and
    # build a vocabulary with 0 = padding (""), 1 = unknown ("[UNK]"), then the
    # tokens from most to least frequent
    counts = Counter(token for element in elements for token in element.split())
    vocab = ["", "[UNK]"] + [token for token, _ in counts.most_common()]
    token_to_index = {token: index for index, token in enumerate(vocab)}

    def vectorize_layer(text):
        return torch.tensor([token_to_index.get(t, 1) for t in text.split()])

    # Tokenize every sequence up front -> LongTensor of shape (n_sequences, SEQ_LEN + 1)
    ds = torch.stack([vectorize_layer(element) for element in elements])
    return ds, vectorize_layer, vocab


notes_seq_ds, notes_vectorize_layer, notes_vocab = create_dataset(notes)
durations_seq_ds, durations_vectorize_layer, durations_vocab = create_dataset(
    durations
)
seq_ds = TensorDataset(notes_seq_ds, durations_seq_ds)  # yields (notes, durations) pairs

In [ ]:
# Display the same example notes and durations converted to ints
example_tokenised_notes = notes_vectorize_layer(example_notes)
example_tokenised_durations = durations_vectorize_layer(example_durations)
print("{:10} {:10}".format("note token", "duration token"))
for i, (note_int, duration_int) in enumerate(
    zip(
        example_tokenised_notes.tolist()[:11],
        example_tokenised_durations.tolist()[:11],
    )
):
    print(f"{note_int:10}{duration_int:10}")

In [ ]:
notes_vocab_size = len(notes_vocab)
durations_vocab_size = len(durations_vocab)

# Display some token:note mappings
print(f"\nNOTES_VOCAB: length = {len(notes_vocab)}")
for i, note in enumerate(notes_vocab[:10]):
    print(f"{i}: {note}")

print(f"\nDURATIONS_VOCAB: length = {len(durations_vocab)}")
# Display some token:duration mappings
for i, note in enumerate(durations_vocab[:10]):
    print(f"{i}: {note}")

## 3. Create the Training Set <a name="create"></a>

In [ ]:
# Create the training set of sequences and the same sequences shifted by one note
def prepare_inputs(notes, durations):
    x = (notes[:, :-1], durations[:, :-1])
    y = (notes[:, 1:], durations[:, 1:])
    return x, y


# The Trainer batches the examples with a data_collator and passes the resulting
# dict to model.forward() as keyword arguments, so the keys match its arguments
def data_collator(examples):
    notes = torch.stack([n for n, _ in examples])
    durations = torch.stack([d for _, d in examples])
    (x_notes, x_durations), (y_notes, y_durations) = prepare_inputs(notes, durations)
    return {
        "note_inputs": x_notes,
        "durations_inputs": x_durations,
        "note_labels": y_notes,
        "duration_labels": y_durations,
    }


ds = ConcatDataset([seq_ds] * DATASET_REPETITIONS)

In [ ]:
example_input_output = data_collator([ds[i] for i in range(BATCH_SIZE)])
print(example_input_output)

## 5. Create the causal attention mask function <a name="causal"></a>

In [ ]:
def causal_attention_mask(n_dest, n_src, device=None):
    # True where a position is allowed to attend. PyTorch broadcasting means we
    # don't need to tile the mask over the batch like the TF version does
    i = torch.arange(n_dest, device=device)[:, None]
    j = torch.arange(n_src, device=device)
    return i >= j - n_src + n_dest


causal_attention_mask(10, 10).int().numpy().T

## 6. Create a Transformer Block layer <a name="transformer"></a>

In [ ]:
class MultiHeadAttention(nn.Module):
    # Keras-style multi-head attention: every head has its own key_dim and the
    # result is projected back to embed_dim. (nn.MultiheadAttention instead splits
    # embed_dim across the heads, which needs EMBEDDING_DIM % N_HEADS == 0.)
    def __init__(self, num_heads, key_dim, embed_dim):
        super().__init__()
        self.num_heads = num_heads
        self.key_dim = key_dim
        self.query = nn.Linear(embed_dim, num_heads * key_dim)
        self.key = nn.Linear(embed_dim, num_heads * key_dim)
        self.value = nn.Linear(embed_dim, num_heads * key_dim)
        self.output = nn.Linear(num_heads * key_dim, embed_dim)

    def forward(self, x, mask):
        batch_size, seq_len, _ = x.shape

        def split_heads(t):  # (B, T, heads * key_dim) -> (B, heads, T, key_dim)
            return t.view(batch_size, seq_len, self.num_heads, self.key_dim).transpose(1, 2)

        q = split_heads(self.query(x))
        k = split_heads(self.key(x))
        v = split_heads(self.value(x))
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.key_dim)
        scores = scores.masked_fill(~mask, float("-inf"))  # hide future tokens
        attention_scores = scores.softmax(dim=-1)  # (B, heads, T, T)
        out = (attention_scores @ v).transpose(1, 2).reshape(batch_size, seq_len, -1)
        return self.output(out), attention_scores


class TransformerBlock(nn.Module):
    def __init__(
        self,
        num_heads,
        key_dim,
        embed_dim,
        ff_dim,
        dropout_rate=DROPOUT_RATE,
    ):
        super().__init__()
        self.attn = MultiHeadAttention(num_heads, key_dim, embed_dim)
        self.dropout_1 = nn.Dropout(dropout_rate)
        self.ln_1 = nn.LayerNorm(embed_dim, eps=1e-6)
        self.ffn_1 = nn.Linear(embed_dim, ff_dim)
        self.ffn_2 = nn.Linear(ff_dim, embed_dim)
        self.dropout_2 = nn.Dropout(dropout_rate)
        self.ln_2 = nn.LayerNorm(embed_dim, eps=1e-6)

    def forward(self, inputs):
        seq_len = inputs.shape[1]
        causal_mask = causal_attention_mask(seq_len, seq_len, inputs.device)
        attention_output, attention_scores = self.attn(inputs, causal_mask)
        attention_output = self.dropout_1(attention_output)
        out1 = self.ln_1(inputs + attention_output)
        ffn_1 = torch.relu(self.ffn_1(out1))
        ffn_2 = self.ffn_2(ffn_1)
        ffn_output = self.dropout_2(ffn_2)
        return (self.ln_2(out1 + ffn_output), attention_scores)

## 7. Create the Token and Position Embedding <a name="embedder"></a>

In [ ]:
class TokenAndPositionEmbedding(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        # He-uniform init as in the book (Keras computes the fan from vocab_size)
        nn.init.kaiming_uniform_(self.token_emb.weight, mode="fan_out", nonlinearity="relu")
        self.pos_emb = SinePositionEncoding()

    def forward(self, x):
        embedding = self.token_emb(x)
        positions = self.pos_emb(embedding)
        return embedding + positions

In [ ]:
tpe = TokenAndPositionEmbedding(notes_vocab_size, 32)
with torch.no_grad():
    token_embedding = tpe.token_emb(example_tokenised_notes)
    position_embedding = tpe.pos_emb(token_embedding)
    embedding = tpe(example_tokenised_notes)
plt.imshow(
    np.transpose(token_embedding.numpy()),
    cmap="coolwarm",
    interpolation="nearest",
    origin="lower",
)
plt.show()
plt.imshow(
    np.transpose(position_embedding.numpy()),
    cmap="coolwarm",
    interpolation="nearest",
    origin="lower",
)
plt.show()
plt.imshow(
    np.transpose(embedding.numpy()),
    cmap="coolwarm",
    interpolation="nearest",
    origin="lower",
)
plt.show()

## 8. Build the Transformer model <a name="transformer_decoder"></a>

In [ ]:
class MusicTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.note_embeddings = TokenAndPositionEmbedding(
            notes_vocab_size, EMBEDDING_DIM // 2
        )
        self.duration_embeddings = TokenAndPositionEmbedding(
            durations_vocab_size, EMBEDDING_DIM // 2
        )
        self.attention = TransformerBlock(
            N_HEADS, KEY_DIM, EMBEDDING_DIM, FEED_FORWARD_DIM
        )
        # No softmax: the heads output logits, nn.CrossEntropyLoss applies it internally
        self.note_outputs = nn.Linear(EMBEDDING_DIM, notes_vocab_size)
        self.duration_outputs = nn.Linear(EMBEDDING_DIM, durations_vocab_size)
        self.loss_fn = nn.CrossEntropyLoss()

    # The Trainer expects the model to compute its own loss when labels are given
    def forward(self, note_inputs, durations_inputs, note_labels=None, duration_labels=None):
        embeddings = torch.cat(
            [
                self.note_embeddings(note_inputs),
                self.duration_embeddings(durations_inputs),
            ],
            dim=-1,
        )
        x, attention_scores = self.attention(embeddings)
        note_logits = self.note_outputs(x)
        duration_logits = self.duration_outputs(x)

        loss = None
        if note_labels is not None:
            # CrossEntropyLoss expects the classes in dim 1: (B, T, vocab) -> (B, vocab, T)
            loss = self.loss_fn(note_logits.transpose(1, 2), note_labels) + self.loss_fn(
                duration_logits.transpose(1, 2), duration_labels
            )
        # The attention scores are returned too, so no separate att_model is needed
        return {
            "loss": loss,
            "note_logits": note_logits,
            "duration_logits": duration_logits,
            "attention_scores": attention_scores,
        }


model = MusicTransformer()

In [ ]:
print(model)
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
if LOAD_MODEL:
    # trainer.save_model() stores the weights as model.safetensors
    from safetensors.torch import load_file

    model.load_state_dict(load_file("./models/model/model.safetensors"))

## 9. Train the Transformer <a name="train"></a>

In [ ]:
# A Trainer callback that generates music at the end of every epoch
class MusicGenerator(TrainerCallback):
    def __init__(self, model, index_to_note, index_to_duration, top_k=10):
        self.model = model
        self.index_to_note = index_to_note
        self.note_to_index = {
            note: index for index, note in enumerate(index_to_note)
        }
        self.index_to_duration = index_to_duration
        self.duration_to_index = {
            duration: index for index, duration in enumerate(index_to_duration)
        }

    def sample_from(self, logits, temperature):
        # softmax(logits / T) is the same as probs ** (1 / T) renormalised
        probs = torch.softmax(logits / temperature, dim=-1)
        return torch.multinomial(probs, 1).item(), probs.cpu().numpy()

    def get_note(self, notes, durations, temperature):
        # Resample if we draw the padding (0) or unknown (1) token
        sample_note_idx = 1
        while sample_note_idx in (0, 1):
            sample_note_idx, note_probs = self.sample_from(
                notes[0][-1], temperature
            )
            sample_note = self.index_to_note[sample_note_idx]

        sample_duration_idx = 1
        while sample_duration_idx in (0, 1):
            sample_duration_idx, duration_probs = self.sample_from(
                durations[0][-1], temperature
            )
            sample_duration = self.index_to_duration[sample_duration_idx]

        new_note = get_midi_note(sample_note, sample_duration)

        return (
            new_note,
            sample_note_idx,
            sample_note,
            note_probs,
            sample_duration_idx,
            sample_duration,
            duration_probs,
        )

    @torch.no_grad()
    def generate(self, start_notes, start_durations, max_tokens, temperature):
        was_training = self.model.training
        self.model.eval()  # switch off dropout
        model_device = next(self.model.parameters()).device

        start_note_tokens = [self.note_to_index.get(x, 1) for x in start_notes]
        start_duration_tokens = [
            self.duration_to_index.get(x, 1) for x in start_durations
        ]
        sample_note = None
        sample_duration = None
        info = []
        midi_stream = music21.stream.Stream()

        midi_stream.append(music21.clef.BassClef())

        for sample_note, sample_duration in zip(start_notes, start_durations):
            new_note = get_midi_note(sample_note, sample_duration)
            if new_note is not None:
                midi_stream.append(new_note)

        while len(start_note_tokens) < max_tokens:
            x1 = torch.tensor([start_note_tokens], device=model_device)
            x2 = torch.tensor([start_duration_tokens], device=model_device)
            outputs = self.model(x1, x2)
            notes, durations, att = (
                outputs["note_logits"],
                outputs["duration_logits"],
                outputs["attention_scores"],
            )

            repeat = True

            while repeat:
                (
                    new_note,
                    sample_note_idx,
                    sample_note,
                    note_probs,
                    sample_duration_idx,
                    sample_duration,
                    duration_probs,
                ) = self.get_note(notes, durations, temperature)

                if (
                    isinstance(new_note, music21.chord.Chord)
                    or isinstance(new_note, music21.note.Note)
                    or isinstance(new_note, music21.note.Rest)
                ) and sample_duration == "0.0":
                    repeat = True
                else:
                    repeat = False

            if new_note is not None:
                midi_stream.append(new_note)

            info.append(
                {
                    "prompt": [start_notes.copy(), start_durations.copy()],
                    "midi": midi_stream,
                    "chosen_note": (sample_note, sample_duration),
                    "note_probs": note_probs,
                    "duration_probs": duration_probs,
                    # attention of the last token over the sequence: (heads, T)
                    "atts": att[0, :, -1, :].cpu().numpy(),
                }
            )
            start_note_tokens.append(sample_note_idx)
            start_duration_tokens.append(sample_duration_idx)
            start_notes.append(sample_note)
            start_durations.append(sample_duration)

            if sample_note == "START":
                break

        self.model.train(was_training)
        return info

    def on_epoch_end(self, args, state, control, **kwargs):
        epoch = round(state.epoch) - 1  # state.epoch counts finished epochs
        info = self.generate(
            ["START"], ["0.0"], max_tokens=GENERATE_LEN, temperature=0.5
        )
        midi_stream = info[-1]["midi"].chordify()
        print(info[-1]["prompt"])
        midi_stream.show()
        midi_stream.write(
            "midi",
            fp=os.path.join(
                "./output",
                "output-" + str(epoch).zfill(4) + ".mid",
            ),
        )

In [ ]:
# TrainingArguments holds the optimizer and training settings
# (the Trainer uses AdamW; with weight_decay=0 it is plain Adam)
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"  # where report_to="tensorboard" writes
training_args = TrainingArguments(
    output_dir="./checkpoint",
    learning_rate=0.001,  # Keras "adam" default
    lr_scheduler_type="constant",  # Keras keeps the learning rate fixed
    max_grad_norm=0.0,  # Keras doesn't clip gradients (the Trainer clips at 1.0 by default)
    per_device_train_batch_size=BATCH_SIZE,
    dataloader_drop_last=True,
    num_train_epochs=EPOCHS,
    logging_strategy="epoch",
    save_strategy="epoch",  # save a checkpoint after every epoch...
    save_total_limit=1,  # ...keeping only the latest one
    report_to="tensorboard",
)

# Tokenize starting prompt
music_generator = MusicGenerator(model, notes_vocab, durations_vocab)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds,
    data_collator=data_collator,
    callbacks=[music_generator],
)

In [ ]:
trainer.train()

In [ ]:
# Save the final model
trainer.save_model("./models/model")

# 3. Generate music using the Transformer

In [ ]:
info = music_generator.generate(
    ["START"], ["0.0"], max_tokens=50, temperature=0.5
)
midi_stream = info[-1]["midi"].chordify()
midi_stream.show()

## Write music to MIDI file

In [ ]:
timestr = time.strftime("%Y%m%d-%H%M%S")
midi_stream.write(
    "midi",
    fp=os.path.join(
        "./output",
        "output-" + timestr + ".mid",
    ),
)

## Note probabilities

In [ ]:
max_pitch = 70
seq_len = len(info)
grid = np.zeros((max_pitch, seq_len), dtype=np.float32)

for j in range(seq_len):
    for i, prob in enumerate(info[j]["note_probs"]):
        try:
            pitch = music21.note.Note(notes_vocab[i]).pitch.midi
            grid[pitch, j] = prob
        except:
            pass  # Don't show key / time signatures

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.set_yticks([int(j) for j in range(35, 70)])
plt.imshow(
    grid[35:70, :],
    origin="lower",
    cmap="coolwarm",
    vmin=-0.5,
    vmax=0.5,
    extent=[0, seq_len, 35, 70],
)
plt.show()

## Attention Plot

In [ ]:
plot_size = 20

att_matrix = np.zeros((plot_size, plot_size))
prediction_output = []
last_prompt = []

In [ ]:
for j in range(plot_size):
    atts = info[j]["atts"].max(axis=0)
    att_matrix[: (j + 1), j] = atts
    prediction_output.append(info[j]["chosen_note"][0])
    last_prompt.append(info[j]["prompt"][0][-1])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(att_matrix, cmap="Greens", interpolation="nearest")

ax.set_xticks(np.arange(-0.5, plot_size, 1), minor=True)
ax.set_yticks(np.arange(-0.5, plot_size, 1), minor=True)
ax.grid(which="minor", color="black", linestyle="-", linewidth=1)
ax.set_xticks(np.arange(plot_size))
ax.set_yticks(np.arange(plot_size))
ax.set_xticklabels(prediction_output[:plot_size])
ax.set_yticklabels(last_prompt[:plot_size])
ax.xaxis.tick_top()

plt.setp(
    ax.get_xticklabels(),
    rotation=90,
    ha="left",
    va="center",
    rotation_mode="anchor",
)
plt.show()